In [174]:
import pandas as pd
import numpy as np


In [175]:
df = pd.read_csv("Churn_Modelling.csv")
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [176]:
df=df.drop(columns=['RowNumber','CustomerId','Surname'],axis=1)
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [177]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,LabelEncoder,StandardScaler


In [178]:
df.isna().sum()

CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

In [179]:
labelen=LabelEncoder()
onehoten=OneHotEncoder()
standasc=StandardScaler()

In [180]:
df['Gender']=labelen.fit_transform(df['Gender'])

In [181]:
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [182]:
geography_a=onehoten.fit_transform(df[['Geography']]).toarray()

In [183]:
geography_a_columns=onehoten.get_feature_names_out()

In [184]:
geography_a_columns

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [185]:
geo_df=pd.DataFrame(data=geography_a,columns=geography_a_columns)
geo_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [186]:
df=pd.concat([df,geo_df],axis=1)
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,France,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,France,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [187]:
df.drop(columns=['Geography'],axis=1,inplace=True)

In [188]:
df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [189]:
import pickle
with open("onehot_encoder_geo.pkl","wb") as file:
 pickle.dump(onehoten,file)

with open("label_encoder_gender.pkl","wb") as file:
 pickle.dump(labelen,file)

In [190]:
x=df.drop(columns=['Exited'],axis=1)
y=df['Exited']

In [191]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.25)

In [192]:
x_train=standasc.fit_transform(x_train)
x_test=standasc.transform(x_test)


In [193]:
with open("scaler.pkl","wb") as file:
    pickle.dump(standasc,file)

# ANN Implementation in Python


In [194]:
import tensorflow as tf
print(tf.__version__)

2.15.0


In [195]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [196]:
#Building the model
model = Sequential([
    Dense(64,activation='relu',input_shape=(x_train.shape[1],)),
    Dense(32,activation='relu'),
    Dense(1,activation='sigmoid')
])


In [197]:
#for forward and backward propagation we have to compile the model without custom optimiser
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])


In [198]:
#with cutsom optimisers
opt= tf.keras.optimizers.Adam(learning_rate=0.01)
loss=tf.keras.losses.binary_crossentropy
model.compile(optimizer=opt,loss=loss,metrics=['accuracy'])


In [199]:
import datetime

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%y%m%d-%H%M%S")

In [200]:
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
tensor_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)
earlystopping_callback=EarlyStopping(monitor="var_loss",patience=5,restore_best_weights=True)

In [201]:
history = model.fit(x_train,y_train,validation_data=(x_test,y_test),epochs=10,callbacks=[tensor_callback,earlystopping_callback])

Epoch 1/10
235/235 [==============================] - 1s 3ms/step - loss: 0.3979 - accuracy: 0.8348 - val_loss: 0.3719 - val_accuracy: 0.8504
Epoch 2/10
235/235 [==============================] - 1s 2ms/step - loss: 0.3543 - accuracy: 0.8533 - val_loss: 0.3544 - val_accuracy: 0.8548
Epoch 3/10
235/235 [==============================] - 1s 3ms/step - loss: 0.3491 - accuracy: 0.8580 - val_loss: 0.3554 - val_accuracy: 0.8508
Epoch 4/10
235/235 [==============================] - 1s 3ms/step - loss: 0.3438 - accuracy: 0.8597 - val_loss: 0.3622 - val_accuracy: 0.8512
Epoch 5/10
235/235 [==============================] - 1s 3ms/step - loss: 0.3411 - accuracy: 0.8620 - val_loss: 0.3490 - val_accuracy: 0.8548
Epoch 6/10
235/235 [==============================] - 1s 2ms/step - loss: 0.3364 - accuracy: 0.8624 - val_loss: 0.3647 - val_accuracy: 0.8428
Epoch 7/10
235/235 [==============================] - 1s 3ms/step - loss: 0.3353 - accuracy: 0.8636 - val_loss: 0.3534 - val_accuracy: 0.8548
Epoch 

In [202]:
model.save("model.h5")

c:\python_envs\global_env\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [203]:
#load tensorboard extension
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [204]:
%tensorboard --logdir logs/fit